In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
from dataclasses import dataclass
from typing import List, Optional
from datetime import datetime
import re

@dataclass
class Answer:
    author: str
    date: str
    content: str
    sub_questions: Optional[List]


@dataclass
class QNA:
    author: bool
    title: str
    views: int
    date: str
    question: str
    answers: List[Answer]


def search_for(term, /, page=1, parsing=False) -> (BeautifulSoup | requests.Response) :
    url = f'https://kin.naver.com/search/list.naver?query={term}&page={page}'
    resp = requests.get(url)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "lxml") if parsing else resp

def fetch_page(html: BeautifulSoup):
    titles = [element.text for element in html.select('ul.basic1 > li > dl > dt > a')]
    links = [element.get('href') for element in html.select('ul.basic1 > li > dl > dt > a')]
    briefs = [element.text for element in html.select('ul.basic1 > li > dl > dd:nth-child(3)')]
    return titles, links, briefs

def fetch_qna(url):
    resp = requests.get(url)
    soup = BeautifulSoup(resp.text, "lxml")
    title = soup.select_one('.endTitleSection')
    if icon := title.select_one('.iconQuestion'):
        icon.decompose()
    title = title.text.strip()
    content = re.sub(r'\s{2,}', ' ', soup.select_one('div.questionDetail').text.strip())[2:]
    return title, content

def fetch_qna2(url):
    resp = requests.get(url)
    soup = BeautifulSoup(resp.text, "lxml")
    title = soup.select_one('.endTitleSection').text
    userinfo = soup.select_one('.userInfo__bullet')
    if len(list(userinfo.children)) == 7:
        views = int(soup.select_one('div.userInfo.userInfo__bullet > span:nth-child(2)').text.replace('조회수', '').strip().replace(',', ''))
        date_str = soup.select_one('div.userInfo.userInfo__bullet > span:nth-child(3)').text
        date = date_str.replace('작성일', '').strip()
    else:
        views = int(soup.select_one('div.userInfo.userInfo__bullet > span:nth-child(1)').text.replace('조회수', '').strip().replace(',', ''))
        date_str = soup.select_one('div.userInfo.userInfo__bullet > span:nth-child(2)').text
        date = date_str.replace('작성일', '').strip()
    question = soup.select_one('div.questionDetail').text
    answers = []
    for answer_area in soup.select('.answerArea'):
        author_name = name if (name := answer_area.select_one('.name')) else 'Unknown'

        content = '\n'.join([elem.text for elem in soup.select('.se-text-paragraph')])
        answer_date = answer_area.select_one('.answerDate')
        if answer_date:
            answer_date = answer_date.text
        additonal_qna = answer_area.select_one('.additionQna')
        sub_questions = None
        if additonal_qna:
            question = additonal_qna.select_one('[data-additional-qna-type="QUESTION"]')
            answer = additonal_qna.select_one( '[data-additional-qna-type="ANSWER"]')

            sub_questions = [question, answer]
        answers.append(Answer(author=author_name, content=content, date=answer_date, sub_questions=sub_questions))
    qna = QNA(author=author_name, title=title, views=views, date=date, question=question, answers=answers)
    return qna
    

if  __name__ == '__main__':
    # term = input('키워드 입력:')
    term = '취업'
    # all_links = []
    # all_briefs = []
    # all_qnas = []
    total_df = pd.DataFrame(columns=['title', 'link'])
    for page_num in range(1, 21):
        html = search_for(term, page=page_num, parsing=True)
        _, links, briefs = fetch_page(html)
        titles = []  
        contents = []
        for link in links:
            title, content = fetch_qna(link)
            titles.append(title)
            contents.append(content)
        df = pd.DataFrame({'title': titles, 'link': links, 'contents': contents})
        total_df = pd.concat([total_df, df], axis=0)
        time.sleep(0.5) 
        # all_titles.extend(titles)
        # all_links.extend(links)
        # all_briefs.extend(briefs)
        # for link in links:
        #     all_qnas.append(fetch_qna(link))
        # time.sleep(2)
             
        

In [13]:
print(total_df)

                           title  \
0              실럽급여 받는 중 취업 햇을 시   
1                산재 취업치료 휴업급여 질문   
2             학사경고 취업,편입에 문제되나요?   
3                   실업급여 수급 중 취업   
4     토스와 오픽 중 어떤 게 더 취업에 유리할까요?   
..                           ...   
5                      실업급여 조기취업   
6    실업급여 구직활동 취업증명서 프린터 해가야하나요?   
7       30대 남자 세무사 사무실 취업 가능합니까?   
8             국가국가장학금 취업연계중점대학유형   
9                      장애인 취업 꿀팁   

                                                 link  \
0   https://kin.naver.com/qna/detail.naver?d1id=6&...   
1   https://kin.naver.com/qna/detail.naver?d1id=6&...   
2   https://kin.naver.com/qna/detail.naver?d1id=11...   
3   https://kin.naver.com/qna/detail.naver?d1id=6&...   
4   https://kin.naver.com/qna/detail.naver?d1id=11...   
..                                                ...   
5   https://kin.naver.com/qna/detail.naver?d1id=6&...   
6   https://kin.naver.com/qna/detail.naver?d1id=6&...   
7   https://kin.naver.com/qna/detail.naver?d1id=4&... 

In [20]:
# index 리셋
total_df.reset_index(drop=True)

,title,link,contents
0,실럽급여 받는 중 취업 햇을 시,https://kin.naver.com/qna/detail.naver?d1id=6&...,실업급여일이 2월 26일이고 현재 4차 진행 중인 상태입니다.그 진행중에 취업을 ...
1,산재 취업치료 휴업급여 질문,https://kin.naver.com/qna/detail.naver?d1id=6&...,에서 일하던중 손가락 열상으로 산재를 승인받았습니다서류상에서는 취업치료가능이라 되어...
2,"학사경고 취업,편입에 문제되나요?",https://kin.naver.com/qna/detail.naver?d1id=11...,"인데 1학기에 학고받을거같습니다(학교를 못나감)그리고 2학기 휴학할건데취업,편입에 ..."
3,실업급여 수급 중 취업,https://kin.naver.com/qna/detail.naver?d1id=6&...,하세요실업급여 2회차까지 받았는데 이번에 취업을 했어요3회차 실업인정일에 취업 신고...
4,토스와 오픽 중 어떤 게 더 취업에 유리할까요?,https://kin.naver.com/qna/detail.naver?d1id=11...,준비하면서 토스와 오픽 중 어떤 시험을 봐야 할지 고민 중입니다. 두 시험 다 장...
...,...,...,...
195,실내디자인학원 비전공자도 취업까지 가능할까요?,https://kin.naver.com/qna/detail.naver?d1id=11...,디자인학원에 관심이 많은 비전공자예요.(__)고등학교를 졸업하고 계약직으로 근무를 ...
196,실업급여 조기취업,https://kin.naver.com/qna/detail.naver?d1id=6&...,실업급여를 1차까지만 받고 취업을해서 조기취업 수당을 받으려고 하는데요. 13일 ...
197,실업급여 구직활동 취업증명서 프린터 해가야하나요?,https://kin.naver.com/qna/detail.naver?d1id=6&...,4차인데요 구직활동 잡코리아에서 두번 했습니다취업증명서 이거를 뽑아서가야하나요 아니...
198,30대 남자 세무사 사무실 취업 가능합니까?,https://kin.naver.com/qna/detail.naver?d1id=4&...,사 시험을 막 시작한 30초 남자 입니다. 1~2년 준비 후 어느 정도 합격각이 보...


In [21]:
# 중복 제거
clean_df = total_df.drop_duplicates(subset=['title'])

In [22]:
# 결측치 처리
clean_df = clean_df.dropna()

In [28]:
# 텍스트 길이 분석 후 처리 : AI 학습에서 짧은 문장은 정보량이 부족 => 제거
# 텍스트 길이 평균보다 큰 내용들만 추출

# 텍스트 길이 평균
mean_len = clean_df['title'].str.len().mean()

# 평균보다 큰 행들만 추출
print()
total_df = clean_df.loc[clean_df['title'].str.len() > mean_len]                                                                           

total_df.to_csv(f'kin_{term}_limit200.csv', sep=',', index=False)

In [ ]:
# 식품 영양 정보를 가져와서 csv로 저장
# 가져올 정보, 식품명, 탄수화물, 지방, 단백질, 나트륨, 총칼로리
label = {
    "FOOD_NM_KR": "식품명",
    "AMT_NUM6": "탄수화물",
    "AMT_NUM4": "지방",
    "AMT_NUM3": "단백질",
    "AMT_NUM13": "나트륨",
    "AMT_NUM1": "총 칼로리",
}
service_key = "b6daf710d771158f04b0554973dee5375617d1c734048279ea826295344a1304"

def search_for_foodinfo(foodname, page_no=1, num_of_rows=5, output_type="json",
                        research_date=None, maker_name=None, food_category=None,
                        item_report_no=None, last_update=None, db_class_name=None):

    endpoint = "https://apis.data.go.kr/1471000/FoodNtrCpntDbInfo02/getFoodNtrCpntDbInq02"

    params = {
        "serviceKey": service_key,
        "pageNo": page_no,
        "numOfRows": num_of_rows,
        "type": output_type,
        "FOOD_NM_KR": foodname,
        "RESEARCH_YMD": research_date,
        "MAKER_NM": maker_name,
        "FOOD_CAT1_NM": food_category,
        "ITEM_REPORT_NO": item_report_no,
        "UPDATE_DATE":	last_update,
        "DB_CLASS_NM": db_class_name,
    }
    resp = requests.get(endpoint, params=params)
    resp.raise_for_status()
    return resp.json()

data = search_for_foodinfo("떡볶이")
foods_info = []
for item in data["body"]["items"]:
    sel = {}
    for key in label.keys():
        sel[label[key]] = item.get(key)
    foods_info.append(sel)
print(foods_info)

[{'식품명': '떡볶이', '탄수화물': '25.96', '지방': '2.96', '단백질': '3.51', '나트륨': '391.000', '총 칼로리': '144.000'}, {'식품명': '떡볶이_채소', '탄수화물': '57.06', '지방': '1.88', '단백질': '3.12', '나트륨': '269.000', '총 칼로리': '258.000'}, {'식품명': '떡볶이_국물맵떡', '탄수화물': '', '지방': '', '단백질': '4.09', '나트륨': '614.000', '총 칼로리': '139.000'}, {'식품명': '떡볶이_까르보떡볶이', '탄수화물': '', '지방': '', '단백질': '4.30', '나트륨': '288.000', '총 칼로리': '180.000'}, {'식품명': '떡볶이_로제닭볶이', '탄수화물': '', '지방': '', '단백질': '7.57', '나트륨': '447.000', '총 칼로리': '176.000'}]
